# Análise de Risco de Vulnerabilidades
Análise das bases tratadas (`data/processed`) para responder:
1. Qual é o nível atual de exposição ao risco?
2. Quais ativos demandam maior atenção?
3. Onde devem ser priorizados os esforços de correção?
4. O processo atual de tratamento está sendo efetivo?

**Premissas**
- Severidade de referência: derivada do CVSS (`severidade_corrigida`), pois a severidade manual diverge do score em cerca de 74% dos registros.
- Exposição = vulnerabilidades **não corrigidas** (`Aberta`, `Em tratamento` e `Aceita`). Risco aceito continua sendo risco.
- Data de referência: 04/10/2026. Correções com data futura são consideradas inconsistentes e ficam fora das métricas de tempo.
- Vulnerabilidades de ativos inexistentes no inventário (`A9999`) entram nos totais, mas ficam fora dos rankings por ativo.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

HOJE = pd.Timestamp('2026-10-04')
ORDEM_SEV = ['Baixa', 'Média', 'Alta', 'Crítica']
ORDEM_CRIT = ['Baixa', 'Média', 'Alta', 'Crítica', 'Não Mapeado']
CORES_SEV = {'Baixa': '#4caf50', 'Média': '#fbc02d', 'Alta': '#fb8c00', 'Crítica': '#d32f2f'}

## 1. Carga e preparação

In [ ]:
df_ativos = pd.read_csv(r'..\data\processed\ativos_limpo.csv', sep=';')
df = pd.read_csv(r'..\data\processed\vulnerabilidades_limpo.csv', sep=';')

df['data_abertura'] = pd.to_datetime(df['data_abertura'])
df['data_correcao'] = pd.to_datetime(df['data_correcao'])
df['flag_data_correcao_futura'] = df['flag_data_correcao_futura'].fillna(False).astype(bool)

# Atributos adicionais do ativo (domínio e localidade não vêm na tabela de vulnerabilidades)
df = df.merge(df_ativos[['ativo_id', 'dominio', 'localidade']], on='ativo_id', how='left')

# Vulnerabilidades sem correção efetiva
df['aberta'] = df['status_vulnerabilidade'] != 'Corrigida'
df['idade_dias'] = np.where(df['aberta'], (HOJE - df['data_abertura']).dt.days, np.nan)

abertas = df[df['aberta']].copy()
print(f'{len(df)} vulnerabilidades | {df.ativo_id.nunique()} ativos | {len(abertas)} sem correção')

## 2. Qual é o nível atual de exposição ao risco?
### 2.1 Indicadores gerais

In [ ]:
kpis = {
    'Total de vulnerabilidades': len(df),
    'Sem correção (exposição)': len(abertas),
    '% sem correção': f"{len(abertas)/len(df):.1%}",
    'Críticas sem correção (CVSS >= 9)': int((abertas.cvss_score >= 9).sum()),
    'Altas ou críticas sem correção (CVSS >= 7)': int((abertas.cvss_score >= 7).sum()),
    'CVSS >= 9 em Produção': int(((abertas.cvss_score >= 9) & (abertas.ambiente == 'Produção')).sum()),
    'CVSS médio do backlog': round(abertas.cvss_score.mean(), 2),
    'Idade mediana do backlog (dias)': int(abertas.idade_dias.median()),
    'Backlog com mais de 90 dias': f"{(abertas.idade_dias > 90).mean():.1%}",
    'Backlog com mais de 180 dias': f"{(abertas.idade_dias > 180).mean():.1%}",
}
pd.Series(kpis, name='valor').to_frame()

### 2.2 Distribuição do backlog por status e severidade

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))

ordem_status = ['Aberta', 'Em tratamento', 'Aceita', 'Corrigida']
cont = df['status_vulnerabilidade'].value_counts().reindex(ordem_status)
sns.barplot(x=cont.index, y=cont.values, palette=['#d32f2f', '#fb8c00', '#fbc02d', '#4caf50'], ax=ax[0])
for i, v in enumerate(cont.values):
    ax[0].text(i, v + 8, f'{v} ({v/len(df):.0%})', ha='center')
ax[0].set_title('Vulnerabilidades por status'); ax[0].set_xlabel(''); ax[0].set_ylabel('Quantidade')

tab = pd.crosstab(df['severidade_corrigida'], df['status_vulnerabilidade']).reindex(index=ORDEM_SEV, columns=ordem_status)
tab.plot(kind='bar', stacked=True, ax=ax[1], color=['#d32f2f', '#fb8c00', '#fbc02d', '#4caf50'])
ax[1].set_title('Status por severidade (CVSS)'); ax[1].set_xlabel(''); ax[1].set_ylabel('Quantidade')
ax[1].tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

### 2.3 Envelhecimento do backlog (aging)
Quanto mais antigo o item, maior a janela de exploração. A tabela cruza faixa de idade e severidade.

In [ ]:
faixas = pd.cut(abertas['idade_dias'], [-1, 30, 60, 90, 180, 400],
                labels=['0-30', '31-60', '61-90', '91-180', '> 180'])
aging = pd.crosstab(faixas, abertas['severidade_corrigida']).reindex(columns=ORDEM_SEV)
display(aging.assign(Total=aging.sum(axis=1)))

aging.plot(kind='bar', stacked=True, figsize=(9, 4.5), color=[CORES_SEV[s] for s in ORDEM_SEV])
plt.title('Backlog por faixa de idade (dias) e severidade'); plt.xlabel('Idade'); plt.ylabel('Vulnerabilidades')
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

### 2.4 Distribuição do CVSS no backlog

In [ ]:
plt.figure(figsize=(9, 4))
sns.histplot(abertas['cvss_score'], bins=20, color='#455a64')
for x, c in [(4, '#fbc02d'), (7, '#fb8c00'), (9, '#d32f2f')]:
    plt.axvline(x, color=c, linestyle='--')
plt.title('Distribuição do CVSS das vulnerabilidades sem correção (linhas: 4, 7 e 9)')
plt.xlabel('CVSS'); plt.ylabel('Quantidade'); plt.tight_layout(); plt.show()

## 3. Quais ativos demandam maior atenção?
Foi construído um **score de risco por ativo**, somando para cada vulnerabilidade sem correção:

`CVSS × peso da criticidade do ativo × fator de ambiente`

- Peso de criticidade: Crítica = 4, Alta = 3, Média = 2, Baixa = 1 (sem classificação = 2).
- Fator de ambiente: Produção = 1,5; demais = 1,0.

Os pesos são premissas de negócio e podem ser ajustados.

In [ ]:
PESO_CRIT = {'Crítica': 4, 'Alta': 3, 'Média': 2, 'Baixa': 1, 'Não Mapeado': 2}
base = abertas[~abertas['flag_ativo_orfao']].copy()
base['peso_crit'] = base['criticidade_ativo'].map(PESO_CRIT)
base['fator_amb'] = np.where(base['ambiente'] == 'Produção', 1.5, 1.0)
base['score'] = base['cvss_score'] * base['peso_crit'] * base['fator_amb']

ranking = (base.groupby('ativo_id')
    .agg(sistema=('sistema', 'first'), criticidade=('criticidade_ativo', 'first'), ambiente=('ambiente', 'first'),
         dominio=('dominio', 'first'), localidade=('localidade', 'first'), status_ativo=('status_ativo', 'first'),
         qtd_abertas=('vuln_id', 'size'), qtd_cvss_7_mais=('cvss_score', lambda s: int((s >= 7).sum())),
         cvss_max=('cvss_score', 'max'), idade_mediana=('idade_dias', 'median'), score=('score', 'sum'))
    .sort_values('score', ascending=False))
ranking['score_acum_%'] = ranking['score'].cumsum() / ranking['score'].sum()
ranking.head(15).round(1)

In [ ]:
top = ranking.head(15).iloc[::-1]
cores = top['ambiente'].map({'Produção': '#c62828', 'Homologação': '#ef6c00', 'Desenvolvimento': '#1976d2'}).fillna('#757575')
plt.figure(figsize=(10, 6))
plt.barh(top.index + ' | ' + top['criticidade'], top['score'], color=cores)
plt.title('Top 15 ativos por score de risco (vermelho = Produção)')
plt.xlabel('Score de risco'); plt.tight_layout(); plt.show()

### 3.1 Concentração do risco
Verifica se poucos ativos concentram a maior parte do risco (curva de Pareto).

In [ ]:
x = np.arange(1, len(ranking) + 1) / len(ranking)
plt.figure(figsize=(8, 4.5))
plt.plot(x, ranking['score_acum_%'].values, color='#c62828', label='Risco acumulado')
plt.plot([0, 1], [0, 1], '--', color='gray', label='Distribuição uniforme')
plt.gca().xaxis.set_major_formatter(PercentFormatter(1)); plt.gca().yaxis.set_major_formatter(PercentFormatter(1))
plt.xlabel('% de ativos (ordenados por score)'); plt.ylabel('% do risco acumulado')
plt.title('Curva de concentração de risco'); plt.legend(); plt.tight_layout(); plt.show()

for n in (10, 20, 40):
    print(f'Top {n} ativos concentram {ranking.score.head(n).sum() / ranking.score.sum():.1%} do score')

### 3.2 Ativos críticos em Produção com vulnerabilidades graves
Combinação de maior impacto: ativos classificados como `Crítica` ou `Alta`, em Produção, com CVSS >= 7 sem correção.

In [ ]:
graves = base[(base.criticidade_ativo.isin(['Crítica', 'Alta'])) & (base.ambiente == 'Produção') & (base.cvss_score >= 7)]
res = (graves.groupby(['ativo_id', 'sistema', 'criticidade_ativo', 'status_ativo'])
       .agg(qtd=('vuln_id', 'size'), cvss_max=('cvss_score', 'max'), idade_max_dias=('idade_dias', 'max'))
       .sort_values(['qtd', 'cvss_max'], ascending=False).reset_index())
print(f'{len(graves)} vulnerabilidades em {res.ativo_id.nunique()} ativos')
res.head(15)

## 4. Onde devem ser priorizados os esforços de correção?
### 4.1 Mapa de calor: criticidade do ativo × severidade da vulnerabilidade
Os quadrantes superiores direitos (ativo crítico e vulnerabilidade crítica) devem ser tratados primeiro.

In [ ]:
mapa = pd.crosstab(base['criticidade_ativo'], base['severidade_corrigida']).reindex(index=ORDEM_CRIT[::-1], columns=ORDEM_SEV).fillna(0)
plt.figure(figsize=(7, 4.5))
sns.heatmap(mapa, annot=True, fmt='.0f', cmap='YlOrRd', cbar=False)
plt.title('Vulnerabilidades sem correção'); plt.xlabel('Severidade (CVSS)'); plt.ylabel('Criticidade do ativo')
plt.tight_layout(); plt.show()

### 4.2 Fila de priorização (P1 a P4)
Regra proposta para organizar a remediação:

| Prioridade | Critério |
|---|---|
| **P1** | CVSS >= 9 em Produção, ou CVSS >= 7 em ativo `Crítica` de Produção |
| **P2** | CVSS >= 7 em Produção, ou CVSS >= 9 em outros ambientes |
| **P3** | Demais casos com CVSS >= 7 |
| **P4** | CVSS < 7 |

In [ ]:
def prioridade(r):
    prod = r.ambiente == 'Produção'
    if (r.cvss_score >= 9 and prod) or (r.cvss_score >= 7 and prod and r.criticidade_ativo == 'Crítica'): return 'P1'
    if (r.cvss_score >= 7 and prod) or r.cvss_score >= 9: return 'P2'
    if r.cvss_score >= 7: return 'P3'
    return 'P4'

abertas['prioridade'] = abertas.apply(prioridade, axis=1)
fila = abertas.groupby('prioridade').agg(
    qtd=('vuln_id', 'size'), ativos=('ativo_id', 'nunique'), idade_mediana=('idade_dias', 'median'),
    aceitas=('status_vulnerabilidade', lambda s: int((s == 'Aceita').sum())),
    em_desativacao=('status_ativo', lambda s: int((s == 'Em desativação').sum())))
fila['% do backlog'] = (fila.qtd / fila.qtd.sum()).map('{:.1%}'.format)
display(fila)

ax = fila['qtd'].plot(kind='bar', color=['#b71c1c', '#e65100', '#f9a825', '#9e9e9e'], figsize=(7, 4))
for i, v in enumerate(fila['qtd']): ax.text(i, v + 5, v, ha='center')
plt.title('Backlog por prioridade'); plt.xlabel(''); plt.ylabel('Vulnerabilidades'); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

### 4.3 Risco por domínio, localidade e ambiente

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.8))
for a, col in zip(ax, ['dominio', 'localidade', 'ambiente']):
    g = base.groupby(col)['score'].sum().sort_values(ascending=False)
    sns.barplot(x=g.values, y=g.index, color='#5c6bc0', ax=a)
    a.set_title(f'Score de risco por {col}'); a.set_xlabel('Score'); a.set_ylabel('')
plt.tight_layout(); plt.show()

In [ ]:
(base.groupby('dominio')
     .agg(qtd=('vuln_id', 'size'), cvss_7_mais=('cvss_score', lambda s: int((s >= 7).sum())), ativos=('ativo_id', 'nunique'), score=('score', 'sum'))
     .assign(score_por_ativo=lambda d: d.score / d.ativos)
     .sort_values('score', ascending=False).round(1))

### 4.4 Ativos em desativação
Vulnerabilidades em ativos marcados como `Em desativação` podem ser eliminadas pelo descomissionamento, sem esforço de correção.

In [ ]:
desat = abertas[abertas['status_ativo'] == 'Em desativação']
print(f'{len(desat)} vulnerabilidades sem correção ({len(desat)/len(abertas):.1%} do backlog) em {desat.ativo_id.nunique()} ativos em desativação')
print(f'Dessas, {int((desat.cvss_score >= 7).sum())} têm CVSS >= 7 e {int((desat.prioridade == "P1").sum())} são P1')
pd.crosstab(abertas['status_ativo'], abertas['prioridade'])

In [ ]:
ranking[ranking.status_ativo == 'Em desativação'].head(10)[['sistema', 'criticidade', 'ambiente', 'qtd_abertas', 'qtd_cvss_7_mais', 'score']].round(1)

### 4.5 Riscos aceitos
Riscos aceitos com CVSS elevado exigem revisão periódica e justificativa formal.

In [ ]:
aceitas = df[df.status_vulnerabilidade == 'Aceita']
print(f'{len(aceitas)} riscos aceitos | CVSS >= 7: {(aceitas.cvss_score >= 7).sum()} | CVSS >= 9: {(aceitas.cvss_score >= 9).sum()}')
print(f'Aceitas em Produção com CVSS >= 7: {((aceitas.cvss_score >= 7) & (aceitas.ambiente == "Produção")).sum()}')

(aceitas.groupby('severidade_corrigida').size().reindex(ORDEM_SEV)
    .to_frame('aceitas')
    .join(df.groupby('severidade_corrigida').size().rename('total'))
    .assign(**{'% aceitas': lambda d: (d.aceitas / d.total).map('{:.1%}'.format)}))

## 5. O processo atual de tratamento está sendo efetivo?
### 5.1 Taxa de correção por severidade, criticidade e ambiente
Se o processo priorizasse por risco, vulnerabilidades mais graves e ativos mais críticos teriam taxa de correção maior.

In [ ]:
df['corrigida'] = df['status_vulnerabilidade'] == 'Corrigida'
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
for a, (col, ordem) in zip(ax, [('severidade_corrigida', ORDEM_SEV), ('criticidade_ativo', ORDEM_CRIT), ('ambiente', ['Produção', 'Homologação', 'Desenvolvimento', 'Não Mapeado'])]):
    t = df.groupby(col)['corrigida'].mean().reindex(ordem).dropna()
    sns.barplot(x=t.index, y=t.values, color='#26a69a', ax=a)
    a.axhline(df['corrigida'].mean(), color='gray', linestyle='--', label='média geral')
    for i, v in enumerate(t.values): a.text(i, v + 0.005, f'{v:.0%}', ha='center')
    a.yaxis.set_major_formatter(PercentFormatter(1)); a.set_ylim(0, 0.4)
    a.set_title(f'Taxa de correção por {col}'); a.set_xlabel(''); a.legend()
plt.tight_layout(); plt.show()
print(f'Taxa de correção geral: {df.corrigida.mean():.1%}')

### 5.2 Tempo de correção
Apenas correções com datas consistentes (`dias_para_correcao` válido).

In [ ]:
ttr = df.dropna(subset=['dias_para_correcao'])
print(f'{len(ttr)} correções válidas de {int(df.corrigida.sum())} marcadas como corrigidas '
      f'({int(df.flag_data_correcao_futura.sum())} com data de correção futura)')
display(ttr.groupby('severidade_corrigida')['dias_para_correcao']
        .agg(['count', 'median', 'mean', lambda s: s.quantile(.9)]).rename(columns={'<lambda_0>': 'p90'})
        .reindex(ORDEM_SEV).round(1))

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
sns.boxplot(data=ttr, x='severidade_corrigida', y='dias_para_correcao', order=ORDEM_SEV, palette=CORES_SEV, ax=ax[0])
ax[0].set_title('Tempo de correção por severidade'); ax[0].set_xlabel(''); ax[0].set_ylabel('Dias')
sns.boxplot(data=ttr, x='origem', y='dias_para_correcao', color='#90a4ae', ax=ax[1])
ax[1].set_title('Tempo de correção por origem'); ax[1].set_xlabel(''); ax[1].set_ylabel('Dias')
plt.tight_layout(); plt.show()

### 5.3 Evolução mensal: entrada versus correção
Compara o volume de vulnerabilidades abertas com as corrigidas em cada mês e o backlog acumulado.

In [ ]:
entradas = df.dropna(subset=['data_abertura']).groupby(df['data_abertura'].dt.to_period('M')).size()
saidas = (df[df.corrigida & ~df.flag_data_correcao_futura].dropna(subset=['data_correcao'])
          .groupby(df['data_correcao'].dt.to_period('M')).size())
mensal = pd.DataFrame({'abertas': entradas, 'corrigidas': saidas}).fillna(0)
mensal = mensal[mensal.index <= '2026-09']      # outubro é parcial
mensal['backlog_acumulado'] = (mensal.abertas - mensal.corrigidas).cumsum()
mensal.index = mensal.index.astype(str)

fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
mensal[['abertas', 'corrigidas']].plot(kind='bar', ax=ax[0], color=['#d32f2f', '#4caf50'])
ax[0].set_title('Vulnerabilidades abertas x corrigidas por mês'); ax[0].set_xlabel(''); ax[0].tick_params(axis='x', rotation=45)
mensal['backlog_acumulado'].plot(marker='o', ax=ax[1], color='#6a1b9a')
ax[1].set_title('Backlog acumulado (aberturas - correções)'); ax[1].set_xlabel('')
ax[1].tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()
mensal.astype(int).T

In [ ]:
# Vulnerabilidades graves (CVSS >= 7) abertas por mês
graves_mes = df[(df.cvss_score >= 7)].dropna(subset=['data_abertura']).groupby(df['data_abertura'].dt.to_period('M')).size()
graves_mes = graves_mes[graves_mes.index <= '2026-09']
graves_mes.index = graves_mes.index.astype(str)
graves_mes.plot(kind='bar', figsize=(9, 4), color='#e65100')
plt.title('Vulnerabilidades com CVSS >= 7 abertas por mês'); plt.xlabel(''); plt.xticks(rotation=45); plt.tight_layout(); plt.show()

## 6. Análises complementares

### 6.1 Origem das vulnerabilidades (ferramenta de detecção)

In [ ]:
(df.groupby('origem')
   .agg(total=('vuln_id', 'size'), cvss_medio=('cvss_score', 'mean'), cvss_7_mais=('cvss_score', lambda s: int((s >= 7).sum())),
        taxa_correcao=('corrigida', 'mean'), ttr_mediano=('dias_para_correcao', 'median'))
   .assign(taxa_correcao=lambda d: d.taxa_correcao.map('{:.1%}'.format))
   .round(2))

### 6.2 Ativos com maior quantidade de vulnerabilidades
A quantidade bruta nem sempre reflete o risco. A comparação com o ranking por score mostra a diferença.

In [ ]:
vol = df[~df.flag_ativo_orfao].groupby('ativo_id').size().rename('total_vulns')
comp = ranking.join(vol).sort_values('total_vulns', ascending=False)
comp.head(10)[['sistema', 'criticidade', 'ambiente', 'total_vulns', 'qtd_abertas', 'score']].round(1)

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=ranking, x='qtd_abertas', y='score', hue='criticidade', hue_order=ORDEM_CRIT,
                size='qtd_cvss_7_mais', sizes=(20, 200), alpha=.7)
plt.title('Quantidade de vulnerabilidades abertas x score de risco por ativo')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left'); plt.tight_layout(); plt.show()

### 6.3 Impacto da qualidade dos dados nos resultados

In [ ]:
qual = pd.Series({
    'Vulnerabilidades em ativos fora do inventário (A9999)': int(df.flag_ativo_orfao.sum()),
    'Data de abertura inválida': int(df.flag_data_abertura_invalida.sum()),
    'Data de correção futura': int(df.flag_data_correcao_futura.sum()),
    'Correção anterior à abertura': int(df.flag_erro_data_sla.sum()),
    'Ativos com criticidade não mapeada': int((df.criticidade_ativo == 'Não Mapeado').sum()),
    'Ativos com ambiente não mapeado': int((df.ambiente == 'Não Mapeado').sum()),
}, name='registros').to_frame()
qual['% da base'] = (qual.registros / len(df)).map('{:.1%}'.format)
qual

## 7. Conclusões
**Exposição**
- A maior parte da base permanece sem correção, incluindo itens com CVSS elevado e itens em Produção. O backlog é antigo: mais da metade tem mais de 90 dias.

**Ativos prioritários**
- O risco está distribuído por muitos ativos, sem concentração em poucos nomes. O ranking (seção 3) e a fila P1 a P4 (seção 4.2) indicam por onde começar.
- Uma parcela relevante do backlog está em ativos em desativação, o que permite reduzir risco por descomissionamento.

**Prioridade de correção**
1. Itens P1 (CVSS >= 9 em Produção ou CVSS >= 7 em ativo crítico de Produção).
2. Riscos aceitos com CVSS alto, que precisam de revisão e justificativa.
3. Ativos em desativação, com prazo de descomissionamento definido.
4. Ativos do topo do ranking de score.

**Efetividade do processo**
- A taxa de correção é baixa e praticamente igual entre severidades, criticidades e ambientes. Isso indica que o tratamento não segue o risco.
- O volume de correções mensais fica abaixo do de aberturas, de modo que o backlog acumulado cresce.
- Não há SLA formal no dado; o tempo de correção das críticas é pouco menor que o das demais.

**Recomendações**
- Adotar severidade baseada em CVSS e contexto (criticidade e ambiente) e definir SLA por prioridade.
- Instituir governança de risco aceito, com aprovador e prazo de revisão.
- Corrigir inconsistências na origem dos dados (datas, inventário de ativos, domínios de valor).
- Monitorar mensalmente backlog por prioridade, aging e taxa de correção.